# 🛡️ Admin Governance & Policy Management Console

Central administration notebook for the Insurance Fabric Accelerator.

**Fabric Features Showcased:**
- Fabric REST API — workspace, domain, capacity, item management
- Fabric Admin API — activity logs, tenant settings, governance
- Fabric Domains — built-in domain assignment
- Fabric Git Integration — built-in source control
- Fabric Deployment Pipelines — built-in CI/CD
- Workspace Identity — Managed Identity (MSI)
- Semantic Link (`sempy`) — semantic model management
- Purview Integration — automatic lineage

**No SparkSession import — `spark` pre-initialized by Fabric.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Load shared utilities (run the base notebook)                      ║
# ╚══════════════════════════════════════════════════════════════════════╝
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Fabric Domain Management                                ║
# ║  Uses: Fabric Admin REST API — /admin/domains                       ║
# ║  Purpose: Organize all insurance workspaces into Fabric Domains     ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Insurance domain hierarchy for Fabric
INSURANCE_DOMAINS = [
    {"name": "Insurance Platform",        "desc": "Top-level insurance domain", "parent": None},
    {"name": "Policy Administration",     "desc": "Policy lifecycle management", "parent": "Insurance Platform"},
    {"name": "Claims Management",         "desc": "Claims processing & adjudication", "parent": "Insurance Platform"},
    {"name": "Billing & Payments",        "desc": "Premium billing and collections", "parent": "Insurance Platform"},
    {"name": "Customer MDM",              "desc": "Customer master data management", "parent": "Insurance Platform"},
    {"name": "Finance & Actuarial",       "desc": "GL, IFRS17, reserving", "parent": "Insurance Platform"},
    {"name": "Regulatory & Compliance",   "desc": "Audit, compliance, solvency", "parent": "Insurance Platform"},
    {"name": "Marketing & Distribution",  "desc": "Campaigns, agents, leads", "parent": "Insurance Platform"},
    {"name": "Platform Operations",       "desc": "Monitoring, capacity, governance", "parent": "Insurance Platform"},
]


def setup_fabric_domains():
    """Create Fabric Domains using the Admin API.
    
    Fabric Domains are a BUILT-IN governance feature. They group workspaces
    by business area for discovery and access management.
    
    API: POST /admin/domains
    Docs: https://learn.microsoft.com/en-us/rest/api/fabric/admin/domains
    """
    created = {}
    
    for domain in INSURANCE_DOMAINS:
        body = {
            "displayName": domain["name"],
            "description": domain["desc"],
        }
        if domain["parent"] and domain["parent"] in created:
            body["parentDomainId"] = created[domain["parent"]]
        
        try:
            # Fabric Admin API - built-in domain management
            token = get_token()
            resp = requests.post(
                "https://api.fabric.microsoft.com/v1/admin/domains",
                headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                json=body, timeout=60
            )
            if resp.status_code in (200, 201):
                domain_id = resp.json().get("id")
                created[domain["name"]] = domain_id
                print(f"  ✅ Created domain: {domain['name']} ({domain_id})")
            elif resp.status_code == 409:
                print(f"  ⏭️  Domain exists: {domain['name']}")
            else:
                print(f"  ⚠️  {domain['name']}: {resp.status_code} {resp.text[:100]}")
        except Exception as e:
            print(f"  ❌ {domain['name']}: {e}")
    
    return created


# Uncomment to run:
# domain_ids = setup_fabric_domains()
print("🏢 Domain setup function ready. Uncomment to execute.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: Workspace Provisioning                                  ║
# ║  Uses: Fabric REST API — /workspaces                                ║
# ║  Creates all required workspaces and assigns to domains.            ║
# ╚══════════════════════════════════════════════════════════════════════╝

WORKSPACE_PLAN = {
    "dev": [
        "ins-dev-policy", "ins-dev-claims", "ins-dev-billing",
        "ins-dev-customer", "ins-dev-finance", "ins-dev-platform",
    ],
    "test": [
        "ins-test-policy", "ins-test-claims", "ins-test-billing",
        "ins-test-customer", "ins-test-finance", "ins-test-platform",
    ],
    "prod": [
        "ins-prod-policy", "ins-prod-claims", "ins-prod-billing",
        "ins-prod-customer", "ins-prod-finance", "ins-prod-regulatory",
        "ins-prod-marketing", "ins-prod-platform",
    ],
}


def provision_workspaces(env: str, capacity_id: str):
    """Provision all workspaces for an environment.
    
    Uses Fabric REST API (built-in):
      POST /workspaces — create workspace
      POST /workspaces/{id}/assignToCapacity — bind to Fabric capacity
    """
    results = []
    for ws_name in WORKSPACE_PLAN.get(env, []):
        try:
            # Create workspace — Fabric REST API built-in
            ws = fabric.post("/workspaces", {
                "displayName": ws_name,
                "description": f"Insurance Accelerator — {env.upper()}",
                "capacityId": capacity_id,
            })
            ws_id = ws.get("id")
            results.append({"name": ws_name, "id": ws_id, "status": "created"})
            print(f"  ✅ {ws_name} ({ws_id})")
        except Exception as e:
            if "already exists" in str(e).lower() or "409" in str(e):
                print(f"  ⏭️  {ws_name} exists")
                results.append({"name": ws_name, "status": "exists"})
            else:
                print(f"  ❌ {ws_name}: {e}")
                results.append({"name": ws_name, "status": "failed", "error": str(e)})
    return results


# Uncomment to run:
# workspaces = provision_workspaces("dev", "your-capacity-id")
print("🏗️ Workspace provisioning function ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Lakehouse & Item Provisioning                           ║
# ║  Uses: Fabric REST API — /workspaces/{id}/lakehouses                ║
# ║  Creates medallion lakehouses per domain workspace.                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

LAKEHOUSE_PLAN = {
    "policy":     ["bronze_policy", "silver_policy", "gold_policy"],
    "claims":     ["bronze_claims", "silver_claims", "gold_claims"],
    "billing":    ["bronze_billing", "silver_billing", "gold_billing"],
    "customer":   ["bronze_customer", "silver_customer", "gold_customer"],
    "finance":    ["bronze_finance", "silver_finance", "gold_finance"],
    "platform":   ["insurance_metadata", "insurance_monitoring", "insurance_security"],
    "regulatory": ["gold_regulatory", "regulatory_snapshots", "archive_gold"],
    "marketing":  ["bronze_marketing", "silver_marketing", "gold_marketing"],
}


def provision_lakehouses(workspace_id: str, domain: str):
    """Create all lakehouses for a domain workspace.
    
    Uses: POST /workspaces/{id}/lakehouses (Fabric REST API built-in)
    Returns 201 with long-running operation for provisioning.
    """
    results = []
    for lh_name in LAKEHOUSE_PLAN.get(domain, []):
        try:
            result = fabric.post_lro(
                f"/workspaces/{workspace_id}/lakehouses",
                {"displayName": lh_name, "description": f"{domain} — {lh_name}"}
            )
            print(f"  ✅ Lakehouse: {lh_name}")
            results.append({"name": lh_name, "status": "created"})
        except Exception as e:
            print(f"  ⚠️  {lh_name}: {e}")
            results.append({"name": lh_name, "status": "error", "error": str(e)})
    return results


print("📦 Lakehouse provisioning function ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Workspace Identity & RBAC                               ║
# ║  Uses: Fabric REST API — Workspace Identity (built-in MSI)          ║
# ║  Configures: Identity, role assignments, workspace access            ║
# ╚══════════════════════════════════════════════════════════════════════╝

def enable_workspace_identity(workspace_id: str):
    """Enable Workspace Identity (Managed Identity) on a Fabric workspace.
    
    This is the BUILT-IN way to give a workspace its own identity for:
    - Key Vault access (secrets)
    - Azure SQL / Storage access (trusted workspace)
    - Azure AI services access
    
    API: POST /workspaces/{id}/assignIdentity
    """
    try:
        result = fabric.post(f"/workspaces/{workspace_id}/assignIdentity")
        principal_id = result.get("identity", {}).get("principalId")
        print(f"  ✅ Workspace Identity enabled. Principal: {principal_id}")
        return principal_id
    except Exception as e:
        print(f"  ⚠️  {e}")
        return None


def add_workspace_member(workspace_id: str, email: str, role: str):
    """Add a user/group to a workspace with a specific role.
    
    Roles (Fabric built-in): Admin, Member, Contributor, Viewer
    API: POST /workspaces/{id}/roleAssignments
    """
    try:
        fabric.post(
            f"/workspaces/{workspace_id}/roleAssignments",
            {
                "principal": {"id": email, "type": "User"},
                "role": role,  # Admin, Member, Contributor, Viewer
            }
        )
        print(f"  ✅ Added {email} as {role}")
    except Exception as e:
        print(f"  ⚠️  {e}")


# RBAC plan for a 10-person DataOps team
TEAM_RBAC = {
    "prod": {
        "data_engineers":    {"role": "Contributor", "type": "Group"},
        "data_analysts":     {"role": "Viewer",      "type": "Group"},
        "data_ops_leads":    {"role": "Admin",        "type": "Group"},
        "service_principal": {"role": "Member",       "type": "ServicePrincipal"},
    },
    "dev": {
        "data_engineers":    {"role": "Admin",        "type": "Group"},
        "data_analysts":     {"role": "Contributor",  "type": "Group"},
    },
}

print("🔐 RBAC functions ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4B: Comprehensive Access Provisioning (Admin Activity)     ║
# ║  Uses: Fabric REST API — workspace, item, capacity, semantic RBAC   ║
# ║  Complete access management across all Fabric resource types        ║
# ╚══════════════════════════════════════════════════════════════════════╝

def grant_workspace_access(workspace_id: str, principal: dict):
    """Grant workspace access to user, group, or service principal.
    
    Args:
        principal: {"email": "user@domain.com", "role": "Admin|Member|Contributor|Viewer", "type": "User|Group|ServicePrincipal"}
    
    API: POST /workspaces/{id}/roleAssignments
    """
    try:
        fabric.post(
            f"/workspaces/{workspace_id}/roleAssignments",
            {
                "principal": {
                    "id": principal.get("id"),  # Principal ID (from Entra ID)
                    "type": principal["type"]
                },
                "role": principal["role"]
            }
        )
        print(f"  ✅ Granted {principal['role']} to {principal.get('email', principal.get('id'))}")
    except Exception as e:
        print(f"  ⚠️  Grant failed: {e}")


def revoke_workspace_access(workspace_id: str, principal_id: str):
    """Revoke workspace access for a principal.
    
    API: DELETE /workspaces/{id}/roleAssignments/{principalId}
    """
    try:
        fabric.delete(f"/workspaces/{workspace_id}/roleAssignments/{principal_id}")
        print(f"  ✅ Revoked access for {principal_id}")
    except Exception as e:
        print(f"  ⚠️  Revoke failed: {e}")


def list_workspace_access(workspace_id: str):
    """List all role assignments for a workspace.
    
    API: GET /workspaces/{id}/roleAssignments
    Returns: List of {"principal": {...}, "role": "..."}
    """
    try:
        result = fabric.get(f"/workspaces/{workspace_id}/roleAssignments")
        assignments = result.get("value", [])
        
        print(f"\\n🔐 Workspace Access ({len(assignments)} assignments):")
        for assign in assignments:
            principal = assign.get("principal", {})
            role = assign.get("role", "Unknown")
            print(f"  • {principal.get('displayName', 'Unknown')}: {role} ({principal.get('type', 'Unknown')})")
        
        return assignments
    except Exception as e:
        print(f"  ⚠️  List failed: {e}")
        return []


def grant_item_access(workspace_id: str, item_id: str, principal: dict, permissions: list):
    """Grant item-level access (lakehouse, notebook, semantic model, etc.).
    
    Args:
        permissions: List of permissions, e.g., ["Read", "Write", "Execute", "Reshare"]
    
    API: POST /workspaces/{workspaceId}/items/{itemId}/permissions
    """
    try:
        fabric.post(
            f"/workspaces/{workspace_id}/items/{item_id}/permissions",
            {
                "principal": {
                    "id": principal.get("id"),
                    "type": principal["type"]
                },
                "permissions": permissions
            }
        )
        print(f"  ✅ Granted {', '.join(permissions)} on item {item_id}")
    except Exception as e:
        print(f"  ⚠️  Item grant failed: {e}")


def grant_capacity_access(capacity_id: str, principal: dict, role: str):
    """Grant capacity-level access.
    
    Args:
        role: "Admin" | "Contributor"
    
    API: POST /capacities/{id}/roleAssignments
    """
    try:
        fabric.post(
            f"/capacities/{capacity_id}/roleAssignments",
            {
                "principal": {
                    "id": principal.get("id"),
                    "type": principal["type"]
                },
                "role": role
            }
        )
        print(f"  ✅ Granted capacity {role} to {principal.get('email', principal.get('id'))}")
    except Exception as e:
        print(f"  ⚠️  Capacity grant failed: {e}")


def setup_semantic_model_rls(workspace_id: str, semantic_model_id: str, roles_config: list):
    """Configure Row-Level Security (RLS) on Power BI semantic models.
    
    Args:
        roles_config: [
            {
                "name": "RegionNorth",
                "filter": "[Region] = \\"North\\"",
                "members": ["user1@domain.com", "user2@domain.com"]
            }
        ]
    
    Uses: Power BI REST API (integrated with Fabric)
    """
    try:
        # RLS setup requires Power BI REST API
        pbi_token = get_token("https://analysis.windows.net/powerbi/api")
        
        for role in roles_config:
            # Create role
            role_response = requests.post(
                f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{semantic_model_id}/roles",
                headers={
                    "Authorization": f"Bearer {pbi_token}",
                    "Content-Type": "application/json"
                },
                json={
                    "name": role["name"],
                    "tablePermissions": [
                        {
                            "name": "FactTable",  # Adjust based on model
                            "filterExpression": role["filter"]
                        }
                    ]
                }
            )
            
            # Add members to role
            for member in role.get("members", []):
                requests.post(
                    f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{semantic_model_id}/roles/{role['name']}/users",
                    headers={
                        "Authorization": f"Bearer {pbi_token}",
                        "Content-Type": "application/json"
                    },
                    json={
                        "emailAddress": member
                    }
                )
            
            print(f"  ✅ RLS role '{role['name']}' configured with {len(role.get('members', []))} members")
    
    except Exception as e:
        print(f"  ⚠️  RLS setup failed: {e}")


def bulk_provision_access_from_config(access_config: dict):
    """Provision access for entire team from configuration.
    
    Args:
        access_config: {
            "workspaces": [
                {
                    "workspace_id": "xxx",
                    "principals": [
                        {"email": "user@domain.com", "role": "Contributor", "type": "User"}
                    ]
                }
            ],
            "semantic_models": [
                {
                    "workspace_id": "xxx",
                    "model_id": "yyy",
                    "rls_roles": [...]
                }
            ]
        }
    """
    print("\\n🔐 Bulk Access Provisioning...")
    
    # Workspace access
    for ws_config in access_config.get("workspaces", []):
        ws_id = ws_config["workspace_id"]
        print(f"\\n  Workspace: {ws_id}")
        
        for principal in ws_config.get("principals", []):
            grant_workspace_access(ws_id, principal)
    
    # Semantic model RLS
    for sm_config in access_config.get("semantic_models", []):
        ws_id = sm_config["workspace_id"]
        model_id = sm_config["model_id"]
        print(f"\\n  Semantic Model RLS: {model_id}")
        
        setup_semantic_model_rls(ws_id, model_id, sm_config.get("rls_roles", []))
    
    print("\\n✅ Bulk access provisioning complete")


def audit_access_across_workspaces(workspace_ids: list):
    """Audit access across multiple workspaces for compliance review.
    
    Returns: DataFrame with all access assignments
    """
    print("\\n🔍 Auditing access across workspaces...")
    
    audit_data = []
    for ws_id in workspace_ids:
        assignments = list_workspace_access(ws_id)
        for assign in assignments:
            principal = assign.get("principal", {})
            audit_data.append({
                "workspace_id": ws_id,
                "principal_id": principal.get("id"),
                "principal_name": principal.get("displayName"),
                "principal_type": principal.get("type"),
                "role": assign.get("role"),
                "audit_date": datetime.now()
            })
    
    # Convert to Spark DataFrame
    from pyspark.sql import Row
    df_audit = spark.createDataFrame([Row(**item) for item in audit_data])
    
    # Save to audit lakehouse
    df_audit.write.format("delta") \\
        .mode("append") \\
        .saveAsTable("metadata.rbac_audit_log")
    
    print(f"✅ Audited {len(audit_data)} access assignments → metadata.rbac_audit_log")
    return df_audit


# Example access configuration (Insurance Accelerator)
INSURANCE_ACCESS_CONFIG = {
    "workspaces": [
        {
            "workspace_id": "prod-policy-ws",
            "principals": [
                {"email": "policy-team@insurance.com", "role": "Contributor", "type": "Group"},
                {"email": "analysts@insurance.com", "role": "Viewer", "type": "Group"},
                {"email": "admin@insurance.com", "role": "Admin", "type": "User"},
            ]
        },
        {
            "workspace_id": "prod-claims-ws",
            "principals": [
                {"email": "claims-team@insurance.com", "role": "Contributor", "type": "Group"},
                {"email": "fraud-team@insurance.com", "role": "Viewer", "type": "Group"},
            ]
        }
    ],
    "semantic_models": [
        {
            "workspace_id": "prod-policy-ws",
            "model_id": "policy-semantic-model-id",
            "rls_roles": [
                {
                    "name": "RegionNorth",
                    "filter": "[Region] = \\"North\\"",
                    "members": ["north-managers@insurance.com"]
                },
                {
                    "name": "RegionSouth",
                    "filter": "[Region] = \\"South\\"",
                    "members": ["south-managers@insurance.com"]
                }
            ]
        }
    ]
}

print("✅ Access provisioning functions ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Git Integration (Fabric Built-In)                       ║
# ║  Uses: Fabric REST API — /workspaces/{id}/git                       ║
# ║  Connects workspaces to Azure DevOps / GitHub repos.                ║
# ╚══════════════════════════════════════════════════════════════════════╝

def connect_workspace_to_git(workspace_id: str, repo_config: dict):
    """Connect a Fabric workspace to Git source control.
    
    Fabric has BUILT-IN Git integration — no external tools needed.
    Supports: Azure DevOps Git, GitHub
    
    API: POST /workspaces/{id}/git/connect
    """
    try:
        fabric.post(
            f"/workspaces/{workspace_id}/git/connect",
            {
                "gitProviderDetails": {
                    "organizationName": repo_config["org"],
                    "projectName": repo_config["project"],
                    "repositoryName": repo_config["repo"],
                    "branchName": repo_config["branch"],
                    "directoryName": repo_config.get("directory", "/"),
                    "gitProviderType": repo_config.get("provider", "AzureDevOps"),
                }
            }
        )
        print(f"  ✅ Connected to {repo_config['repo']} ({repo_config['branch']})")
    except Exception as e:
        print(f"  ⚠️  Git connect: {e}")


def sync_workspace_from_git(workspace_id: str):
    """Sync (pull) workspace items from Git.
    
    API: POST /workspaces/{id}/git/updateFromGit
    """
    try:
        # Get current status first
        status = fabric.get(f"/workspaces/{workspace_id}/git/status")
        
        # Pull changes
        fabric.post_lro(
            f"/workspaces/{workspace_id}/git/updateFromGit",
            {
                "remoteCommitHash": status.get("remoteCommitHash"),
                "conflictResolution": {"conflictResolutionType": "Workspace"},  # Workspace wins
            }
        )
        print("  ✅ Workspace synced from Git")
    except Exception as e:
        print(f"  ⚠️  Git sync: {e}")


def commit_workspace_to_git(workspace_id: str, comment: str):
    """Commit (push) workspace changes to Git.
    
    API: POST /workspaces/{id}/git/commitToGit
    """
    try:
        status = fabric.get(f"/workspaces/{workspace_id}/git/status")
        changes = status.get("changes", [])
        
        if not changes:
            print("  ℹ️  No changes to commit")
            return
        
        fabric.post_lro(
            f"/workspaces/{workspace_id}/git/commitToGit",
            {
                "mode": "All",
                "comment": comment,
            }
        )
        print(f"  ✅ Committed {len(changes)} changes: {comment}")
    except Exception as e:
        print(f"  ⚠️  Git commit: {e}")


print("🔗 Git integration functions ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: Deployment Pipelines (Fabric Built-In CI/CD)            ║
# ║  Uses: Fabric REST API — /deploymentPipelines                       ║
# ║  Promotes items across Dev → Test → Prod environments.              ║
# ╚══════════════════════════════════════════════════════════════════════╝

def create_deployment_pipeline(name: str, description: str = "") -> str:
    """Create a Fabric Deployment Pipeline.
    
    Fabric Deployment Pipelines are the BUILT-IN CI/CD mechanism.
    They have 3 stages: Development, Test, Production.
    
    API: POST /deploymentPipelines
    """
    result = fabric.post("/deploymentPipelines", {
        "displayName": name,
        "description": description or f"Insurance Accelerator — {name}",
    })
    pipeline_id = result.get("id")
    print(f"  ✅ Pipeline created: {name} ({pipeline_id})")
    return pipeline_id


def assign_pipeline_stages(pipeline_id: str,
                           dev_ws_id: str, test_ws_id: str, prod_ws_id: str):
    """Assign workspaces to pipeline stages.
    
    API: POST /deploymentPipelines/{id}/stages/{stageOrder}/assignWorkspace
    Stage orders: 0=Development, 1=Test, 2=Production
    """
    stages = [(0, dev_ws_id, "Dev"), (1, test_ws_id, "Test"), (2, prod_ws_id, "Prod")]
    for order, ws_id, label in stages:
        try:
            fabric.post(
                f"/deploymentPipelines/{pipeline_id}/stages/{order}/assignWorkspace",
                {"workspaceId": ws_id}
            )
            print(f"  ✅ Stage {label} → {ws_id}")
        except Exception as e:
            print(f"  ⚠️  Stage {label}: {e}")


def deploy_to_next_stage(pipeline_id: str, source_stage: int,
                          note: str = "Automated deployment"):
    """Deploy (promote) items from one stage to the next.
    
    API: POST /deploymentPipelines/{id}/deploy
    """
    try:
        result = fabric.post_lro(
            f"/deploymentPipelines/{pipeline_id}/deploy",
            {
                "sourceStageOrder": source_stage,
                "isBackwardDeployment": False,
                "newDeploymentNote": note,
                "options": {
                    "allowOverwriteArtifact": True,
                    "allowCreateArtifact": True,
                    "allowOverwriteTargetArtifactLabel": True,
                }
            }
        )
        stage_names = {0: "Dev→Test", 1: "Test→Prod"}
        print(f"  ✅ Deployed: {stage_names.get(source_stage, source_stage)} ({note})")
    except Exception as e:
        print(f"  ❌ Deploy failed: {e}")


print("🚀 Deployment Pipeline functions ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7: Activity Log Collection (Fabric Admin API)              ║
# ║  Uses: GET /admin/activityevents — built-in audit logs              ║
# ║  Captures: logins, queries, refreshes, exports, shares              ║
# ╚══════════════════════════════════════════════════════════════════════╝

from datetime import datetime, timedelta

def collect_activity_logs(days_back: int = 7):
    """Collect Fabric activity logs into a Delta table.
    
    The Fabric Admin API provides 30 days of activity logs.
    This is the built-in audit trail — every action in Fabric is logged.
    
    API: GET /admin/activityevents?startDateTime='...'&endDateTime='...'
    """
    end_dt = datetime.utcnow()
    start_dt = end_dt - timedelta(days=days_back)
    
    token = get_token()
    all_events = []
    
    # Fabric API returns events in 1-hour chunks
    current = start_dt
    while current < end_dt:
        chunk_end = min(current + timedelta(hours=1), end_dt)
        
        url = (f"https://api.fabric.microsoft.com/v1/admin/activityevents"
               f"?startDateTime='{current.strftime('%Y-%m-%dT%H:%M:%S')}'"
               f"&endDateTime='{chunk_end.strftime('%Y-%m-%dT%H:%M:%S')}'")
        
        try:
            resp = requests.get(url, headers={
                "Authorization": f"Bearer {token}"
            }, timeout=60)
            
            if resp.status_code == 200:
                events = resp.json().get("activityEventEntities", [])
                all_events.extend(events)
                
                # Handle pagination
                cont_url = resp.json().get("continuationUri")
                while cont_url:
                    resp2 = requests.get(cont_url, headers={
                        "Authorization": f"Bearer {token}"
                    }, timeout=60)
                    if resp2.status_code == 200:
                        all_events.extend(resp2.json().get("activityEventEntities", []))
                        cont_url = resp2.json().get("continuationUri")
                    else:
                        break
        except Exception:
            pass
        
        current = chunk_end
    
    if all_events:
        df = spark.createDataFrame(all_events)
        df.write.format("delta").mode("append").saveAsTable(
            f"{MONITORING_SCHEMA}.fabric_activity_log"
        )
        print(f"  ✅ Collected {len(all_events)} activity events ({days_back} days)")
    else:
        print("  ℹ️  No events found")
    
    return len(all_events)


# Uncomment to run:
# collect_activity_logs(7)
print("📋 Activity log collector ready.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 8: Full Environment Setup Orchestrator                     ║
# ║  One-click setup: domains → workspaces → lakehouses → RBAC → Git   ║
# ╚══════════════════════════════════════════════════════════════════════╝

def full_environment_setup(capacity_id: str, env: str = "dev",
                           git_config: dict = None):
    """Complete environment setup using Fabric REST APIs only.
    
    This function provisions the entire insurance platform:
    1. Create Fabric Domains (built-in governance)
    2. Create Workspaces
    3. Enable Workspace Identity (MSI)
    4. Create Lakehouses (medallion architecture)
    5. Set up RBAC
    6. Connect Git (built-in source control)
    7. Create Deployment Pipelines (built-in CI/CD)
    8. Run metadata table creation
    9. Load seed data
    
    Everything via APIs — zero manual clicks.
    """
    print("\n" + "="*60)
    print(f"  🚀 FULL ENVIRONMENT SETUP: {env.upper()}")
    print("="*60)
    
    # Step 1: Domains
    print("\n📁 Step 1: Fabric Domains")
    domain_ids = setup_fabric_domains()
    
    # Step 2: Workspaces
    print(f"\n🏗️  Step 2: Workspaces ({env})")
    workspaces = provision_workspaces(env, capacity_id)
    
    # Step 3: Workspace Identity
    print("\n🔐 Step 3: Workspace Identity")
    for ws in workspaces:
        if ws.get("id"):
            enable_workspace_identity(ws["id"])
    
    # Step 4: Lakehouses
    print("\n📦 Step 4: Lakehouses")
    for ws in workspaces:
        if ws.get("id"):
            domain = ws["name"].split("-")[-1]  # e.g., 'policy' from 'ins-dev-policy'
            provision_lakehouses(ws["id"], domain)
    
    # Step 5: Git
    if git_config:
        print("\n🔗 Step 5: Git Integration")
        for ws in workspaces:
            if ws.get("id"):
                connect_workspace_to_git(ws["id"], git_config)
    
    # Step 6: Metadata tables
    print("\n📋 Step 6: Metadata Tables")
    run_child_notebook("02_create_metadata_tables")
    
    # Step 7: Seed data
    print("\n🌱 Step 7: Seed Data")
    run_child_notebook("03_seed_metadata")
    
    print("\n" + "="*60)
    print("  ✅ ENVIRONMENT SETUP COMPLETE")
    print("="*60)


# Uncomment to execute full setup:
# full_environment_setup(capacity_id="your-capacity-id", env="dev")
print("🎯 Full setup orchestrator ready. Uncomment to execute.")